# EasyMart RAG Chatbot — Mini Eval (Session 3)

**เป้าหมาย:** วัดคุณภาพ retrieval layer ด้วย precision@3 และ recall@3

ขั้นตอน:
1. เตรียม ground truth (10 คำถาม + chunk ที่ถูก)
2. รัน retrieval สำหรับแต่ละคำถาม top-k=3
3. คำนวณ precision@3 และ recall@3
4. plot histogram ของ similarity score ของ top-1

In [ ]:
# ติดตั้ง dependencies (รันครั้งแรก หรือถ้ายังไม่ได้ติดตั้ง)
# !pip install sentence-transformers faiss-cpu numpy matplotlib

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import faiss

# นำเข้า chunk_markdown จาก app.py (ไม่นำเข้า load_index เพราะใช้ @st.cache_resource)
sys.path.insert(0, os.path.abspath('.'))
from app import chunk_markdown

print('✅ Import สำเร็จ')

## Step 1: โหลด KB และสร้าง FAISS Index (Standalone สำหรับ Notebook)

In [ ]:
# โหลด menu_kb.md
kb_path = 'menu_kb.md'
with open(kb_path, 'r', encoding='utf-8') as f:
    content = f.read()

# Split เป็น chunks
chunks = chunk_markdown(content)
print(f'✅ โหลดสำเร็จ: {len(chunks)} chunks')

# โหลด embedding model
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print('✅ โหลด SentenceTransformer สำเร็จ')

# Encode chunks
embeddings = model.encode(chunks, convert_to_numpy=True, normalize_embeddings=True)

# สร้าง FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype(np.float32))

print(f'✅ สร้าง FAISS Index สำเร็จ (dim={dimension}, {index.ntotal} vectors)')
print('\nตัวอย่าง chunks 5 อันแรก:')
for i, c in enumerate(chunks[:5]):
    print(f'  [{i}] {c[:80]}...' if len(c) > 80 else f'  [{i}] {c}')

## Step 2: เตรียม Ground Truth (10 คำถาม)

แต่ละคำถามกำหนด keyword ที่ต้องปรากฎใน chunk ที่ retrieve ได้ (manual labeling)

In [ ]:
# Ground truth: 10 คำถาม + keyword ที่ควรอยู่ใน retrieved chunk
ground_truth = [
    {
        'question': 'EasyMart เปิดกี่โมง?',
        'relevant_keywords': ['08:00', '22:00', 'เปิด']
    },
    {
        'question': 'ค่าส่งเท่าไหร่?',
        'relevant_keywords': ['30 บาท', 'ค่าส่ง', 'delivery']
    },
    {
        'question': 'ซื้อครบเท่าไหรถึงส่งฟรี?',
        'relevant_keywords': ['200 บาท', 'ส่งฟรี', 'โปรโมชั่น']
    },
    {
        'question': 'นมสดราคาเท่าไหร่?',
        'relevant_keywords': ['45 บาท', 'นมสด', 'Dutch Mill']
    },
    {
        'question': 'มีสินค้าที่มี gluten ไหม?',
        'relevant_keywords': ['gluten', 'ขนมปัง', 'บิสกิต']
    },
    {
        'question': 'ชำระเงินด้วยวิธีอะไรได้บ้าง?',
        'relevant_keywords': ['พร้อมเพย์', 'บัตรเครดิต', 'ชำระ']
    },
    {
        'question': 'มีโปรวันจันทร์ไหม?',
        'relevant_keywords': ['จันทร์', '10%', 'เครื่องดื่ม']
    },
    {
        'question': 'ออเดอร์ขั้นต่ำเท่าไหร่?',
        'relevant_keywords': ['50 บาท', 'ขั้นต่ำ']
    },
    {
        'question': 'คืนสินค้าได้ไหม?',
        'relevant_keywords': ['คืน', '24 ชั่วโมง']
    },
    {
        'question': 'ช็อกโกแลตราคาเท่าไหร่?',
        'relevant_keywords': ['35 บาท', 'ช็อกโกแลต', 'KitKat']
    },
]

print(f'✅ Ground truth: {len(ground_truth)} คำถาม')

## Step 3: รัน Retrieval และคำนวณ Precision@3 / Recall@3

In [ ]:
K = 3  # top-k

def chunk_is_relevant(chunk: str, keywords: list) -> bool:
    """ตรวจว่า chunk มี keyword ที่กำหนดอย่างน้อย 1 ตัว"""
    chunk_lower = chunk.lower()
    return any(kw.lower() in chunk_lower for kw in keywords)

results = []
top1_scores = []

for item in ground_truth:
    question = item['question']
    relevant_keywords = item['relevant_keywords']

    # Encode query
    q_emb = model.encode([question], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)

    # Search FAISS
    scores, indices = index.search(q_emb, K)
    retrieved_chunks = [chunks[idx] for idx in indices[0] if 0 <= idx < len(chunks)]
    top1_score = float(scores[0][0]) if len(scores[0]) > 0 else 0.0
    top1_scores.append(top1_score)

    # Precision@K = จำนวน chunk ที่ retrieve แล้วถูก / K
    correct_count = sum(1 for c in retrieved_chunks if chunk_is_relevant(c, relevant_keywords))
    precision_at_k = correct_count / K

    # Recall@K = 1.0 ถ้า retrieve ได้ relevant chunk อย่างน้อย 1 อัน
    recall_at_k = 1.0 if correct_count > 0 else 0.0

    results.append({
        'question': question,
        'retrieved_chunks': retrieved_chunks,
        'top1_score': top1_score,
        'precision@3': precision_at_k,
        'recall@3': recall_at_k,
        'correct_count': correct_count,
    })

    print(f"Q: {question}")
    for j, rc in enumerate(retrieved_chunks, 1):
        mark = '✅' if chunk_is_relevant(rc, relevant_keywords) else '❌'
        print(f"  [{j}] {mark} {rc[:70]}")
    print(f"   → Top-1 Score: {top1_score:.4f} | Precision@3: {precision_at_k:.2f} | Recall@3: {recall_at_k:.2f}")
    print()

## Step 4: สรุปผล Average Precision@3 และ Recall@3

In [ ]:
avg_precision = np.mean([r['precision@3'] for r in results])
avg_recall = np.mean([r['recall@3'] for r in results])

print('=' * 52)
print('📊 EasyMart RAG Eval Results')
print('=' * 52)
print(f'Average Precision@3 : {avg_precision:.4f}  ({avg_precision*100:.1f}%)')
print(f'Average Recall@3    : {avg_recall:.4f}  ({avg_recall*100:.1f}%)')
print()
print(f"{'คำถาม':<38} {'P@3':>5} {'R@3':>5} {'Score':>8}")
print('-' * 60)
for r in results:
    q_short = r['question'][:35] + '...' if len(r['question']) > 35 else r['question']
    print(f"{q_short:<38} {r['precision@3']:>5.2f} {r['recall@3']:>5.2f} {r['top1_score']:>8.4f}")

## Step 5: Plot Histogram ของ Top-1 Similarity Score

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('EasyMart RAG Retrieval Evaluation', fontsize=14, fontweight='bold')

# --- Plot 1: Histogram ของ Top-1 Similarity Scores ---
ax1 = axes[0]
ax1.hist(top1_scores, bins=8, color='steelblue', edgecolor='white', alpha=0.85)
ax1.axvline(np.mean(top1_scores), color='tomato', linestyle='--', linewidth=2,
            label=f'Mean = {np.mean(top1_scores):.3f}')
ax1.set_xlabel('Cosine Similarity Score (Top-1)', fontsize=11)
ax1.set_ylabel('Frequency', fontsize=11)
ax1.set_title('Distribution of Top-1 Similarity Scores', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# --- Plot 2: Bar chart ของ Precision@3 ต่อคำถาม ---
ax2 = axes[1]
questions_short = [r['question'][:14] + '...' if len(r['question']) > 14 else r['question']
                   for r in results]
precisions = [r['precision@3'] for r in results]
bar_colors = ['#2ecc71' if p >= 0.34 else '#e74c3c' for p in precisions]
ax2.bar(range(len(questions_short)), precisions, color=bar_colors, edgecolor='white', alpha=0.85)
ax2.set_xticks(range(len(questions_short)))
ax2.set_xticklabels(questions_short, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('Precision@3', fontsize=11)
ax2.set_title(f'Precision@3 per Question\n(Avg = {avg_precision:.2f})', fontsize=12)
ax2.set_ylim(0, 1.2)
ax2.axhline(avg_precision, color='navy', linestyle='--', linewidth=1.5,
            label=f'Avg = {avg_precision:.2f}')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('eval_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ บันทึกกราฟเป็น eval_results.png')

## Reflection

**วิเคราะห์ผลลัพธ์:**

| Metric | ค่าที่ได้ | ความหมาย |
|---|---|---|
| Precision@3 | - | % ของ 3 chunk ที่ retrieve มาที่ตรงกับคำถาม |
| Recall@3 | - | สัดส่วนคำถามที่พบ relevant chunk ใน top-3 |

**คำถามที่ retrieve ไม่ดี (Precision@3 = 0):**
- คำถามเหล่านี้มักใช้คำศัพท์ที่ไม่ตรงกับ chunk ใน KB
- หรือคำถามมี context หลายหัวข้อ ทำให้ retrieval สับสน

**แนวทางปรับปรุง:**
1. เพิ่มขนาด chunk / ปรับ chunking strategy
2. เพิ่ม top-k เป็น 5 หรือ 7
3. เพิ่มข้อมูลใน `menu_kb.md` ให้ครอบคลุมมากขึ้น
4. ใช้ hybrid search (BM25 + semantic)